<a href="https://colab.research.google.com/github/fborrasumh/bioestadistica/blob/main/cuadernos/Tema_070_Ficheros_y_formatos_de_datos_sanitarios_CSV_y_JSON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# <font color='steelblue'>Tema 70. Gestión de ficheros y formatos de intercambio de datos sanitarios: CSV y JSON</font>

**Bioestadística (1964) · Grado en Medicina · 2.º curso · Curso 2027-2028**

**Módulo 2. Programando con IA**

**Autor:** Fernando Borrás Rocher · Universidad Miguel Hernández de Elche · ORCID [0000-0002-5519-4573](https://orcid.org/0000-0002-5519-4573)

**Duración:** 50 minutos de clase, más el estudio de los apartados de ampliación

**Práctica:** [unidad 70 de la app Bioestadística](https://fborrasumh.github.io/bioestadistica/)

**Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es). Todos los pacientes y datos de este cuaderno son sintéticos.

*Antes de trabajar, guarda tu copia: **Archivo → Guardar una copia en Drive**.*

## <font color='steelblue'>Configuración del cuaderno</font>

Ejecuta esta celda al empezar. Instala el módulo de preguntas interactivas y prepara el cuaderno.

**Para dar la clase con el cuaderno:** usa **Entorno de ejecución → Ejecutar todo** antes de empezar, para que se dibujen los gráficos. Las celdas con borde azul son las diapositivas; el desarrollo de cada bloque está debajo y se puede plegar con la flecha de su título. El índice (☰, a la izquierda) sirve para saltar de un bloque a otro.

In [ ]:
#@title Ejecutar configuración
%pip install -q jupyterquiz==2.9.6.4
from jupyterquiz import display_quiz
print("Cuaderno configurado")

## <font color='steelblue'>Objetivos de aprendizaje</font>

Al terminar esta sesión serás capaz de:

1. Crear, leer y ampliar ficheros de texto desde Python.
2. Guardar y leer una tabla de pacientes en formato **CSV**.
3. Reconocer los problemas habituales de los CSV en español: separador, coma decimal y codificación.
4. Convertir datos de Python a **JSON** y viceversa, y extraer información de un JSON anidado.
5. Reconocer la estructura de un recurso del estándar sanitario **HL7 FHIR**.
6. Entender la respuesta de una API, que llega en formato JSON.

## <font color='steelblue'>Plan de la sesión</font>

| Bloque | Minutos |
| --- | --- |
| 1. Caso clínico de partida | 5 |
| 2. Conceptos clave | 10 |
| 3. Demostración: ficheros, CSV y JSON | 20 |
| 4. Práctica en la app Bioestadística | 10 |
| 5. Autoevaluación | 5 |

## <font color='steelblue'>1. Caso clínico de partida (5 min)</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">El CSV que se abrió mal</p><img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_070_csv_roto.png" width="780" alt="El CSV que se abrió mal"><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>Separador «;», coma decimal y tildes en UTF-8: si no se indican al leer, los datos se estropean.</li><li>Los datos viajan en <b>formatos de fichero</b>; conocerlos evita perder información.</li></ul></div>

#### Desarrollo para el estudio

Un grupo de investigación de un hospital participa en un **registro multicéntrico** de insuficiencia cardiaca. Cada mes debe enviar los datos de sus pacientes (seudonimizados) en un fichero **CSV**. Al mismo tiempo, el laboratorio del hospital entrega los resultados analíticos al sistema de historia clínica en **JSON**, siguiendo el estándar internacional **HL7 FHIR**.

Una residente abre el CSV en una hoja de cálculo y descubre que todas las columnas aparecen juntas en una sola, que los pesos han perdido los decimales y que las tildes se ven como símbolos extraños.

Los datos no se transmiten en el vacío: viajan en **formatos de fichero**. Conocerlos evita perder información por el camino, y es imprescindible para el Tema 80, en el que un modelo de IA nos responderá en JSON.

## <font color='steelblue'>2. Conceptos clave (10 min)</font>

### <font color='steelblue'>2.1 Ficheros de texto</font>

Un fichero se abre con `open(nombre, modo, encoding="utf-8")`:

| Modo | Significado | Cuidado |
| --- | --- | --- |
| `"r"` | Leer | Error si el fichero no existe |
| `"w"` | Escribir | **Borra** el contenido anterior |
| `"a"` | Añadir al final | Conserva lo que había |

Se usa siempre con `with`, que cierra el fichero automáticamente al terminar. La codificación **UTF-8** representa correctamente tildes, eñes y símbolos como «µ» o «°».

En Colab, los ficheros creados se guardan en una carpeta temporal visible en el panel **Archivos** y **se borran al cerrar la sesión**. Para conservarlos hay que descargarlos o guardarlos en Drive.

### <font color='steelblue'>2.2 CSV</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">CSV: una tabla escrita como texto</p><img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_070_csv.png" width="780" alt="CSV: una tabla escrita como texto"><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>Una línea por paciente; los campos separados por un carácter.</li><li>La primera línea suele ser la cabecera con los nombres de las variables.</li><li>Un campo vacío es un dato ausente, no un cero.</li></ul></div>

#### Desarrollo para el estudio

**CSV** (*comma-separated values*) es una tabla en texto plano: una fila por línea y las columnas separadas por un carácter. Es el formato más usado para intercambiar datos tabulares.

```
codigo,edad,sexo,peso_kg,fevi_pct
IC-001,72,M,81.5,35
IC-002,68,F,64.0,
```

En España, muchas hojas de cálculo usan el **punto y coma** como separador y la **coma** como separador decimal. Por eso conviene saber siempre qué separador tiene un fichero antes de leerlo. Una celda vacía indica un **dato ausente**.

### <font color='steelblue'>2.3 JSON</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">JSON: datos anidados, como un resultado de laboratorio HL7 FHIR</p><img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_070_json.png" width="780" alt="JSON: datos anidados, como un resultado de laboratorio HL7 FHIR"><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>Diccionarios dentro de diccionarios, y listas: el formato de las API y de la historia clínica electrónica.</li><li>Para llegar a un dato se sigue el camino de claves: obs["valueQuantity"]["value"].</li></ul></div>

#### Desarrollo para el estudio

**JSON** (*JavaScript Object Notation*) representa datos estructurados en texto. Su sintaxis es casi idéntica a la de los diccionarios y listas de Python, y admite datos **anidados**: diccionarios dentro de listas dentro de diccionarios.

| JSON | Python |
| --- | --- |
| objeto `{ }` | diccionario `dict` |
| array `[ ]` | lista `list` |
| `"texto"` | `str` |
| `12`, `12.5` | `int`, `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

Es el formato de las **API** web, de los modelos de IA y del estándar sanitario **HL7 FHIR**, con el que se intercambian historias clínicas, resultados de laboratorio y prescripciones.

## <font color='steelblue'>3. Demostración: ficheros, CSV y JSON (20 min)</font>

### <font color='steelblue'>3.1 Escribir y leer un fichero de texto</font>

In [ ]:
informe = """INFORME DE ALTA (paciente sintético IC-001)
Diagnóstico principal: insuficiencia cardiaca con fracción de eyección reducida.
Tratamiento al alta: sacubitrilo/valsartán, bisoprolol, dapagliflozina.
Revisión en consulta de Cardiología en 4 semanas.
"""

with open("informe_alta.txt", "w", encoding="utf-8") as fichero:
    fichero.write(informe)

with open("informe_alta.txt", "r", encoding="utf-8") as fichero:
    contenido = fichero.read()

print(contenido)

Con el modo `"a"` se añade una línea sin borrar lo anterior. Leer con `.readlines()` devuelve una lista con una línea por elemento.

In [ ]:
with open("informe_alta.txt", "a", encoding="utf-8") as fichero:
    fichero.write("Alergias: no conocidas.\n")

with open("informe_alta.txt", "r", encoding="utf-8") as fichero:
    lineas = fichero.readlines()

print("Número de líneas:", len(lineas))
print("Última línea:", lineas[-1].strip())

### <font color='steelblue'>3.2 Buscar información en el texto</font>

In [ ]:
texto = contenido.lower()
for farmaco in ["bisoprolol", "furosemida", "dapagliflozina"]:
    print(f"{farmaco}: {'sí' if farmaco in texto else 'no'} aparece en el informe")

### <font color='steelblue'>3.3 Guardar una tabla en CSV</font>

El módulo `csv` escribe una lista de diccionarios como tabla. `DictWriter` usa las claves como nombres de columna.

In [ ]:
import csv

pacientes = [
    {"codigo": "IC-001", "edad": 72, "sexo": "M", "peso_kg": 81.5, "fevi_pct": 35},
    {"codigo": "IC-002", "edad": 68, "sexo": "F", "peso_kg": 64.0, "fevi_pct": None},
    {"codigo": "IC-003", "edad": 81, "sexo": "M", "peso_kg": 70.2, "fevi_pct": 28},
]

with open("registro_ic.csv", "w", encoding="utf-8", newline="") as fichero:
    escritor = csv.DictWriter(fichero, fieldnames=["codigo", "edad", "sexo", "peso_kg", "fevi_pct"])
    escritor.writeheader()
    escritor.writerows(pacientes)

print(open("registro_ic.csv", encoding="utf-8").read())

Observa que el dato ausente de IC-002 queda como una celda vacía.

### <font color='steelblue'>3.4 Leer un CSV</font>

`DictReader` lee cada fila como un diccionario. **Todo llega como texto**: hay que convertir los números y tratar las celdas vacías.

In [ ]:
with open("registro_ic.csv", encoding="utf-8") as fichero:
    lector = csv.DictReader(fichero)
    for fila in lector:
        edad = int(fila["edad"])
        fevi = float(fila["fevi_pct"]) if fila["fevi_pct"] != "" else None
        print(fila["codigo"], edad, "años · FEVI:", fevi)

### <font color='steelblue'>3.5 El CSV «a la española»</font>

Este es el aspecto de un CSV exportado desde una hoja de cálculo configurada en español: punto y coma como separador y coma decimal.

In [ ]:
csv_espanol = """codigo;edad;peso_kg
IC-004;77;68,4
IC-005;59;92,1
"""
with open("registro_es.csv", "w", encoding="utf-8") as fichero:
    fichero.write(csv_espanol)

with open("registro_es.csv", encoding="utf-8") as fichero:
    lector = csv.DictReader(fichero, delimiter=";")
    for fila in lector:
        peso = float(fila["peso_kg"].replace(",", "."))
        print(fila["codigo"], peso, "kg")

Si se leyera con el separador por defecto (la coma), cada línea se partiría por la coma decimal y los datos quedarían desordenados. En el Tema 100, Pandas resolverá esto con dos parámetros: `sep=";"` y `decimal=","`.

### <font color='steelblue'>3.6 JSON: de Python a texto y de vuelta</font>

In [ ]:
import json

paciente = {"codigo": "IC-001", "edad": 72, "diabetes": True, "fevi_pct": None,
            "farmacos": ["bisoprolol", "dapagliflozina"]}

texto_json = json.dumps(paciente, ensure_ascii=False, indent=2)
print(texto_json)

Observa la traducción: `True` pasa a `true` y `None` a `null`. Con `json.loads()` se recupera el diccionario.

In [ ]:
recuperado = json.loads(texto_json)
print(type(recuperado), recuperado["farmacos"][0])

In [ ]:
with open("paciente.json", "w", encoding="utf-8") as fichero:
    json.dump(paciente, fichero, ensure_ascii=False, indent=2)

with open("paciente.json", encoding="utf-8") as fichero:
    print(json.load(fichero)["edad"])

### <font color='steelblue'>3.7 Un resultado de laboratorio en HL7 FHIR</font>

FHIR organiza la información clínica en **recursos** (Patient, Observation, MedicationRequest…). Este es un recurso *Observation* simplificado con una glucemia, codificada con el sistema internacional **LOINC**. Para llegar al valor hay que recorrer los niveles de anidamiento.

In [ ]:
observacion_fhir = """
{
  "resourceType": "Observation",
  "status": "final",
  "code": {
    "coding": [{"system": "http://loinc.org", "code": "2339-0",
                "display": "Glucose [Mass/volume] in Blood"}]
  },
  "subject": {"reference": "Patient/IC-001"},
  "effectiveDateTime": "2027-10-04T08:15:00+02:00",
  "valueQuantity": {"value": 112, "unit": "mg/dL"}
}
"""

obs = json.loads(observacion_fhir)
codigo = obs["code"]["coding"][0]
print("Prueba:", codigo["display"], f"(LOINC {codigo['code']})")
print("Paciente:", obs["subject"]["reference"])
print("Resultado:", obs["valueQuantity"]["value"], obs["valueQuantity"]["unit"])

### <font color='steelblue'>3.8 La respuesta de una API</font>

Cuando, en el Tema 80, enviemos una pregunta a un modelo de IA, la respuesta llegará como un JSON parecido a este (simplificado). Lo que interesa está en lo más profundo: el texto generado y el número de tokens consumidos, que determina el coste.

In [ ]:
respuesta_api = json.loads("""
{
  "modelo": "modelo-ejemplo",
  "candidatos": [
    {"contenido": {"texto": "El IMC de 25,6 kg/m² corresponde a sobrepeso según la OMS."},
     "motivo_fin": "STOP"}
  ],
  "uso": {"tokens_entrada": 18, "tokens_salida": 16}
}
""")

print(respuesta_api["candidatos"][0]["contenido"]["texto"])
uso = respuesta_api["uso"]
print("Tokens totales:", uso["tokens_entrada"] + uso["tokens_salida"])

### <font color='steelblue'>Ampliación: subir ficheros y usar Google Drive</font>

Para trabajar con un fichero de tu ordenador:

```python
from google.colab import files
subidos = files.upload()      # abre un diálogo para elegir el fichero
```

Para leer y guardar en tu Google Drive, de forma que los ficheros no se borren al cerrar la sesión:

```python
from google.colab import drive
drive.mount("/content/drive")
# Tus ficheros quedan en /content/drive/MyDrive/
```

Recuerda: **nunca** subas a Colab ficheros con datos de pacientes reales.

### <font color='steelblue'>Ampliación: errores frecuentes</font>

**Abrir en modo `"w"` un fichero que se quería conservar:** se borra su contenido sin aviso.

**Olvidar `encoding="utf-8"`:** las tildes se leen como símbolos extraños (por ejemplo, «Ã³» en lugar de «ó»).

**No indicar el separador correcto** al leer un CSV con punto y coma.

**Usar los valores del CSV sin convertirlos:** `"72" + "5"` es `"725"`, no `77`.

**Confundir `json.dumps()` y `json.dump()`:** el primero devuelve un texto; el segundo escribe en un fichero.

**`FileNotFoundError`:** el fichero no existe en la carpeta actual, o la sesión se reinició y los ficheros temporales se borraron.

## <font color='steelblue'>4. Práctica en la app Bioestadística (10 min)</font>

Abre la **[unidad 70 de la app Bioestadística](https://fborrasumh.github.io/bioestadistica/)** y completa sus misiones:

1. **Misión básica (sin asistente):** identificar el separador, la coma decimal y los datos ausentes de varios CSV.
2. **Misión clínica (con pistas):** leer un CSV de un registro sintético y calcular la edad media y el número de pacientes sin FEVI registrada.
3. **Misión reto (con asistente):** extraer de un recurso FHIR *Observation* el valor, la unidad y la fecha, y verificar el código que proponga el asistente con un segundo recurso distinto.

Al terminar, exporta el **informe de la unidad**.

## <font color='steelblue'>5. Autoevaluación (5 min)</font>

Responde sin asistente y sin ejecutar código.

In [ ]:
#@title Pregunta interactiva
preguntas = [{'question': 'Abres un fichero que ya contiene datos con open("datos.csv", "w"). ¿Qué ocurre con '
              'el contenido anterior?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Se borra',
               'correct': True,
               'feedback': 'Correcto: el modo "w" crea el fichero de nuevo. Para añadir se usa '
                           '"a".'},
              {'answer': 'Se conserva y lo nuevo se añade al final',
               'correct': False,
               'feedback': 'Eso hace el modo "a".'},
              {'answer': 'Se produce un error',
               'correct': False,
               'feedback': 'No hay error; simplemente se sobrescribe.'},
              {'answer': 'Se crea una copia de seguridad',
               'correct': False,
               'feedback': 'Python no hace copias automáticas.'}]},
 {'question': 'Al leer un CSV con csv.DictReader, la edad aparece como "72". ¿De qué tipo es?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'str',
               'correct': True,
               'feedback': 'Correcto: todo lo que se lee de un CSV llega como texto y hay que '
                           'convertirlo.'},
              {'answer': 'int',
               'correct': False,
               'feedback': 'Hay que convertirlo con int() para que sea un entero.'},
              {'answer': 'float',
               'correct': False,
               'feedback': 'csv no convierte los números automáticamente.'},
              {'answer': 'dict',
               'correct': False,
               'feedback': 'La fila completa es un diccionario; cada valor es texto.'}]},
 {'question': '¿En qué se convierte el valor null de un JSON al leerlo con json.loads()?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'None',
               'correct': True,
               'feedback': 'Correcto: null es el valor nulo de JSON y equivale a None.'},
              {'answer': '0', 'correct': False, 'feedback': 'null no es un cero.'},
              {'answer': '"null"',
               'correct': False,
               'feedback': 'json.loads lo traduce a None, no a un texto.'},
              {'answer': 'False',
               'correct': False,
               'feedback': 'null y false son valores distintos en JSON.'}]},
 {'question': 'Un CSV exportado de una hoja de cálculo en español contiene la línea '
              'IC-004;77;68,4. ¿Qué parámetros necesitas al leerlo?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Separador punto y coma y convertir la coma decimal en punto',
               'correct': True,
               'feedback': 'Correcto: el separador es ; y los decimales vienen con coma.'},
              {'answer': 'Ninguno: csv lo detecta automáticamente',
               'correct': False,
               'feedback': 'Por defecto csv usa la coma como separador.'},
              {'answer': 'Separador coma',
               'correct': False,
               'feedback': 'Partiría el peso 68,4 en dos columnas.'},
              {'answer': 'encoding="latin-1"',
               'correct': False,
               'feedback': 'La codificación afecta a las tildes, no al separador.'}]},
 {'question': 'En el recurso FHIR del apartado 3.7, ¿qué expresión obtiene la unidad del '
              'resultado?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'obs["valueQuantity"]["unit"]',
               'correct': True,
               'feedback': 'Correcto: la unidad está dentro de valueQuantity.'},
              {'answer': 'obs["unit"]',
               'correct': False,
               'feedback': 'unit no está en el primer nivel del recurso.'},
              {'answer': 'obs["code"]["unit"]',
               'correct': False,
               'feedback': 'code describe la prueba, no el resultado.'},
              {'answer': 'obs.unit',
               'correct': False,
               'feedback': 'A los diccionarios se accede con corchetes y la clave.'}]}]
display_quiz(preguntas)

## <font color='steelblue'>Ideas clave</font>

- Los ficheros se abren con `with open(..., encoding="utf-8")`; el modo `"w"` sobrescribe.
- CSV es el formato de las tablas; todo se lee como texto y hay que comprobar separador y coma decimal.
- JSON es el formato de los datos estructurados y anidados, de las API y del estándar HL7 FHIR.
- Para llegar a un dato en un JSON anidado se encadenan claves e índices.

## <font color='steelblue'>Glosario</font>

| Término | Definición |
| --- | --- |
| Codificación (UTF-8) | Forma de representar los caracteres, incluidas tildes y símbolos |
| CSV | Formato de tabla en texto plano con columnas separadas por un carácter |
| Separador | Carácter que separa las columnas de un CSV (coma, punto y coma, tabulador) |
| JSON | Formato de texto para datos estructurados y anidados |
| Serializar | Convertir datos de Python en texto (por ejemplo, JSON) para guardarlos o enviarlos |
| HL7 FHIR | Estándar internacional de intercambio de información sanitaria |
| LOINC | Sistema internacional de códigos para pruebas de laboratorio y observaciones clínicas |
| Seudonimizar | Sustituir los datos identificativos por un código, como IC-001 |

## <font color='steelblue'>Para saber más</font>

- [Tutorial de Python: lectura y escritura de archivos](https://docs.python.org/es/3/tutorial/inputoutput.html#reading-and-writing-files)
- [Documentación del módulo csv](https://docs.python.org/es/3/library/csv.html)
- [Documentación del módulo json](https://docs.python.org/es/3/library/json.html)
- [Especificación de HL7 FHIR](https://hl7.org/fhir/)

---
*Tema 70 · Bioestadística (1964) · Grado en Medicina · Universidad Miguel Hernández de Elche · Fernando Borrás Rocher · CC BY 4.0*